# 20 · Continual Learning & Testing in Production — Closing the Loop

**Part 2 · Machine Learning Systems** · *Prerequisites: 19* · *Book mapping: DMLS ch. 9*

Notebook 19 ended with alarms ringing. Two questions remain, and they are *the* two
questions of production ML: **when and how do you retrain?** and — since offline metrics
already failed to predict the drift — **how do you prove a new model is better in the
world, without burning the world down while you check?**

**Learning objectives.**

1. Compare retraining policies (never / scheduled / monitor-triggered) on nb 19's
   drifting world, and measure the hard floor that label delay puts under *any* policy.
2. Walk the deployment-testing ladder: **shadow → canary → A/B**, building each.
3. Reproduce the **peeking problem** — how daily significance-checking manufactures
   false discoveries — and fix it.
4. Implement **Thompson sampling** and see when bandits beat fixed experiments (and when
   they don't belong).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=20)

## 1. Retraining policies on a drifting world

Same world as nb 19 (covariate shift day 40, concept drift day 80, labels mature 7 days
late). Three policies, honestly simulated — a retrain may only use labels that have
**matured** by the day it runs:

In [ ]:
DAYS, PER_DAY, LABEL_DELAY = 120, 400, 7

def sample_day(day, r):
    center = np.array([0.0, 0.0]) if day < 40 else np.array([1.0, -0.5])
    X = r.normal(0, 1, (PER_DAY, 2)) + center
    if day < 80:
        logits = X[:, 0] + 0.6 * X[:, 1] - 0.15 * X[:, 0] ** 2 - 0.3
    else:
        logits = -0.4 * X[:, 0] + 1.1 * X[:, 1] - 0.15 * X[:, 0] ** 2 - 0.3
    y = (logits + r.normal(0, 0.6, PER_DAY) > 0).astype(int)
    return X, y

world = [sample_day(d, np.random.default_rng(1000 + d)) for d in range(DAYS)]

def train_on(day_range):
    X = np.vstack([world[d][0] for d in day_range])
    y = np.concatenate([world[d][1] for d in day_range])
    return LogisticRegression().fit(X, y)

def run_policy(policy: str, window: int = 20, period: int = 15,
               alarm_drop: float = 0.12, cooldown: int = 8):
    """Simulate 120 days. Retrains use only days whose labels matured
    (day <= today - LABEL_DELAY). Returns (daily accuracy, retrain days)."""
    model = train_on(range(0, 30))
    acc = np.zeros(DAYS)
    retrains, last_retrain = [], 30
    baseline = None
    for d in range(DAYS):
        acc[d] = model.score(*world[d])
        matured_through = d - LABEL_DELAY
        if d < 35 or matured_through < 30:
            continue
        observed = acc[max(30, matured_through - 10):matured_through + 1].mean()
        if baseline is None:
            baseline = observed
        retrain = False
        if policy == "periodic" and (d - last_retrain) >= period:
            retrain = True
        elif policy == "triggered" and observed < baseline - alarm_drop \
                and (d - last_retrain) >= cooldown:
            retrain = True
        if retrain:
            train_days = range(max(0, matured_through - window), matured_through + 1)
            model = train_on(train_days)
            retrains.append(d)
            last_retrain = d
    return acc, retrains

results = {p: run_policy(p) for p in ("never", "periodic", "triggered")}
for name, (acc, retrains) in results.items():
    print(f"{name:10s} mean accuracy (days 30+): {acc[30:].mean():.3f}   "
          f"retrains: {len(retrains)} {retrains}")

acc_never = results["never"][0][30:].mean()
acc_periodic = results["periodic"][0][30:].mean()
acc_triggered = results["triggered"][0][30:].mean()
assert acc_periodic > acc_never + 0.05 and acc_triggered > acc_never + 0.05, \
    "any retraining beats fossilization on a drifting world"
assert len(results["triggered"][1]) < len(results["periodic"][1]), \
    "the monitor-triggered policy spends fewer retrains for comparable accuracy"

# The floor no policy can break: days 80..(80+delay+1) are lost to label delay.
for name, (acc, _) in results.items():
    assert acc[81:88].mean() < 0.6, f"{name}: no labels, no fix - physics"
print("\nno policy recovered before labels matured: label delay floors everything")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
for (name, (acc, retrains)), color in zip(results.items(),
                                          ["#9e9e9e", "#1565c0", "#2e7d32"]):
    ax.plot(acc, color=color, label=name, alpha=0.85)
    for rd in retrains:
        ax.axvline(rd, color=color, alpha=0.25, linestyle=":")
ax.axvline(80, color="#c62828", alpha=0.6)
ax.set_xlabel("day")
ax.set_ylabel("daily accuracy")
ax.set_title("Retraining policies (dotted = retrain events; red = concept drift)")
ax.legend()
plt.show()

Read the plot's fine print: the first triggered retrain *after* day 80 barely helps —
the only matured labels it could use were mostly pre-drift. Recovery needs post-drift
labels to mature *and accumulate*. So retraining speed = label delay + data accumulation
+ pipeline time — and shrinking that loop (streaming labels, faster pipelines, nb 21's
orchestration) is worth more than any modeling cleverness on the same cadence.
**Stateless vs stateful**: we retrained from scratch each time (stateless — simple,
reproducible, immune to inherited corruption); stateful fine-tuning from the previous
checkpoint is cheaper per update and shines at high frequency, but a poisoned update
propagates forever and reproducibility gets subtle (exercise 1).

## 2. The testing-in-production ladder

Offline evaluation (nb 17) compares models on yesterday's questions. Production asks
different ones: does it crash? does it meet latency (nb 18)? do its *actions* change user
behavior? The ladder climbs risk gradually:

### 2.1 Shadow deployment — zero user risk

The challenger scores real traffic; its predictions are logged, never served. When
labels mature, compare champion and challenger on identical, *live* data:

In [ ]:
champion = train_on(range(0, 30))                       # the fossil
challenger = None
shadow_scores = {"A": [], "B": [], "y": []}

for d in range(95, 110):                                # post-drift traffic
    if challenger is None:
        # (In real life the challenger was retrained by the pipeline; here: on
        # matured post-drift days.)
        challenger = train_on(range(88, 95))
    X, y = world[d]
    shadow_scores["A"].append(champion.predict_proba(X)[:, 1])
    shadow_scores["B"].append(challenger.predict_proba(X)[:, 1])   # logged only!
    shadow_scores["y"].append(y)

y_all = np.concatenate(shadow_scores["y"])
auc_A = roc_auc_score(y_all, np.concatenate(shadow_scores["A"]))
auc_B = roc_auc_score(y_all, np.concatenate(shadow_scores["B"]))
print(f"shadow verdict on live traffic - champion AUC {auc_A:.3f}, "
      f"challenger AUC {auc_B:.3f} (no user ever saw B)")
assert auc_B > auc_A + 0.1, "the shadow phase already proves B on real, current data"

Shadow catches: crashes, latency, feature skew (nb 16 — B's features come from the real
serving path), and prediction quality on matured labels. What it *cannot* catch: any
effect that requires users to **experience** B's decisions (a different ranking changes
what people click). For that, someone must actually be served — enter the canary.

### 2.2 Canary release — bounded blast radius

Serve B to 1%, watch **guardrail metrics** (error rate, latency, conversion), ramp only
on green. Simulate a genuinely bad model reaching a 4-stage ramp with an automatic
rollback rule:

In [ ]:
DAILY_USERS, BASE_CONV, BAD_CONV = 50_000, 0.050, 0.035   # B quietly hurts conversion

def canary_rollout(stages=(0.01, 0.05, 0.25, 1.0), z_crit: float = 2.5):
    """Each stage: expose share*users to B for a day, z-test conversion vs
    control. Roll back on breach. Returns users ever exposed to B."""
    exposed = 0
    for share in stages:
        n_b = int(DAILY_USERS * share)
        n_a = DAILY_USERS - n_b
        conv_b = rng.binomial(n_b, BAD_CONV) / n_b
        conv_a = rng.binomial(n_a, BASE_CONV) / n_a
        exposed += n_b
        pooled = (conv_a * n_a + conv_b * n_b) / (n_a + n_b)
        z = (conv_a - conv_b) / np.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
        print(f"  stage {share:>4.0%}: B conversion {conv_b:.3f} vs A {conv_a:.3f} "
              f"(z={z:.1f})" + ("  -> ROLLBACK" if z > z_crit else "  -> ramp"))
        if z > z_crit:
            return exposed
    return exposed

print("canary rollout of a bad model:")
exposed_canary = canary_rollout()
exposed_yolo = DAILY_USERS                               # 'just ship it' exposure
print(f"users exposed to the bad model - canary: {exposed_canary:,}, "
      f"full launch: {exposed_yolo:,}")
assert exposed_canary <= 0.06 * exposed_yolo, \
    "the ramp caught it early: blast radius ~ the stage size, not the user base"

(The rollback itself is nb 18 exercise 5's one-command flip — canary is that machinery
plus a statistical tripwire. Note the z threshold is deliberately strict: guardrails at
1% have little data, so they catch *disasters*, not subtleties. Subtleties need...)

### 2.3 A/B testing — measurement, and the peeking trap

The definitive instrument: randomize users to A or B, compare the business metric with a
significance test. The math is Stats 101; the production failure is *procedural*:
dashboards update live, and humans stop experiments **the moment the p-value dips below
0.05**. Simulate that habit on A/A tests — two identical variants, where every
"discovery" is by definition false:

In [ ]:
def z_pvalue(conv_a, n_a, conv_b, n_b) -> float:
    pooled = (conv_a * n_a + conv_b * n_b) / (n_a + n_b)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    if se == 0:
        return 1.0
    return float(2 * (1 - norm.cdf(abs(conv_a - conv_b) / se)))

def aa_experiment(seed, peek: bool, n_per_day=1_000, days=20):
    r = np.random.default_rng(seed)
    ca = cb = na = nb = 0
    for day in range(days):
        ca += r.binomial(n_per_day, BASE_CONV)
        cb += r.binomial(n_per_day, BASE_CONV)          # SAME rate: any 'win' is noise
        na += n_per_day
        nb += n_per_day
        if peek and z_pvalue(ca / na, na, cb / nb, nb) < 0.05:
            return True                                  # 'significant!' ship it!
    return z_pvalue(ca / na, na, cb / nb, nb) < 0.05     # fixed-horizon decision

n_sims = 300
fp_peeking = np.mean([aa_experiment(s, peek=True) for s in range(n_sims)])
fp_fixed = np.mean([aa_experiment(s, peek=False) for s in range(n_sims)])
print(f"false-discovery rate on A/A tests - peek daily & stop: {fp_peeking:.0%}, "
      f"fixed horizon: {fp_fixed:.0%}")
assert fp_peeking > 2.5 * max(fp_fixed, 0.02), \
    "peeking turns the promised 5% error rate into ~a quarter of experiments 'winning'"
assert fp_fixed < 0.09, "the fixed-horizon test keeps its promise"

Twenty daily looks = twenty chances for noise to cross the line; the p-value's guarantee
only holds for **one** pre-committed look. Fixes: commit the horizon in advance (power
analysis tells you how long — exercise 2), or use methods *designed* for continuous
monitoring (sequential tests, alpha-spending, always-valid p-values). And keep A/A tests
running permanently — they are the unit tests of your experimentation platform.

### 2.4 Bandits — when finding out and cashing in merge

A fixed A/B assigns half of everyone to the loser for the whole test. **Bandits** shift
traffic toward whichever arm *looks* better as evidence accumulates — paying less
**regret** (forgone conversions) for the same learning. Thompson sampling: keep a Beta
posterior per arm; for each user, sample from each posterior, serve the argmax:

In [ ]:
P_ARMS = np.array([0.050, 0.062])                       # B truly better
HORIZON = 20_000

def run_assignment(strategy: str, seed: int = 3) -> float:
    """Returns regret: conversions lost vs always serving the best arm."""
    r = np.random.default_rng(seed)
    wins = np.ones(2)                                    # Beta(1,1) priors
    losses = np.ones(2)
    pulls = np.zeros(2)
    for t in range(HORIZON):
        if strategy == "ab":
            arm = t % 2                                  # rigid 50/50
        else:                                            # thompson
            arm = int(np.argmax(r.beta(wins, losses)))
        pulls[arm] += 1
        if r.random() < P_ARMS[arm]:
            wins[arm] += 1
        else:
            losses[arm] += 1
    return float(pulls @ (P_ARMS.max() - P_ARMS))

regret_ab = np.mean([run_assignment("ab", s) for s in range(10)])
regret_ts = np.mean([run_assignment("thompson", s) for s in range(10)])
print(f"expected conversions sacrificed to learning - A/B 50/50: {regret_ab:.0f}, "
      f"Thompson sampling: {regret_ts:.0f}")
assert regret_ts < 0.5 * regret_ab, \
    "the bandit funnels traffic to the winner while still learning"

Why not bandits for everything, then? Because the A/B's rigidity *is its virtue* as a
measuring instrument: fixed randomization gives unbiased, easily-interpretable estimates
of the difference (and of segment effects, guardrails, long-term metrics); bandits give
you the *decision* cheaply but a biased, adaptive sample. Rules of thumb: bandits for
many short-lived variants with fast feedback (headlines, creatives, exploration budgets
for nb 19's feedback loops); A/B when you need the *number*, when feedback is slow
(bandits with 30-day labels barely adapt), or when switching users between arms is
itself harmful.

## 3. Exercises

1. **Stateful risks.** Fine-tuning from the previous checkpoint daily is 20× cheaper
   than full retrains. Describe three distinct ways it goes wrong (think: nb 14's
   pipeline bug week, nb 19's recurring seasonality, reproducing "the model as of
   March 3rd" — nb 17 ex. 5), and a hybrid schedule that keeps most of the savings.
2. **Power arithmetic.** Baseline conversion 5%; you care about +0.5pp absolute. With
   α=0.05 and 80% power, n per arm ≈ 16·p(1−p)/δ² — compute it, and the test duration at
   4,000 users/day split 50/50. What does the answer say about A/B testing on small
   products, and which §2 tools remain?
3. **Guardrails.** Your ranking A/B improves clicks +2%. Name four guardrail metrics
   that must *not* regress for the launch to proceed, including one from nb 18, one from
   nb 17's slices, and one long-term metric an A/B struggles to see at all.
4. **The unit of randomization.** Why is per-request randomization wrong for a UI
   experiment (same user, both variants), and when is per-user wrong too (marketplace
   sellers, social networks)? Name the interference pattern in each.
5. **Window wars.** After nb 19 ex. 1's December seasonality, argue sliding-window vs
   full-history retraining data — and the third option that beats both (season as a
   *feature*, or per-season models). What data structure from nb 14 makes all three
   equally easy to try?

### Solutions

**1.** (a) *Poisoned lineage*: the bad-pipeline week's gradients live in every later
checkpoint; stateless retrains forget bad data once it exits the window — stateful never
does (without surgery). (b) *One-way drift*: fine-tuning tracks the current season and
slowly forgets December; when December returns (recurring drift), the model meets it as
a stranger. (c) *Reproducibility*: "the March 3rd model" is now a chain of 60
checkpoints + 60 data slices — bit-exact rebuilds get heroic. Hybrid: daily fine-tunes +
weekly stateless full retrain from the immutable lake (a checkpoint "compaction",
nb 03's pattern in MLOps costume).

**2.** n ≈ 16 × 0.05·0.95 / 0.005² ≈ **30,400 per arm** → 60,800 total → ~15 days at
4k/day. Small products can't detect small effects in reasonable time — their options:
test bigger swings only, use more sensitive metrics (per-user continuous outcomes beat
binary conversion), or accept bandit-style decisions without precise measurement.

**3.** p99 latency (nb 18 — a slower model "wins" clicks then loses users); error/timeout
rate; revenue or conversions-per-user (clicks can cannibalize purchases); weak-slice
metrics (nb 17 — the +2% may be −10% for new mobile users); and a long-horizon metric
like 30-day retention — which the A/B window barely sees: hold a small long-term holdout
cohort out of all experiments as the canary for slow damage.

**4.** Per-request: a user sees layout A, refreshes into layout B — inconsistent
experience contaminates behavior (and their sessions straddle arms: unit-of-analysis
error inflates significance). Per-user fails under **interference**: marketplace arms
share inventory (B's buyers deplete stock A's buyers see — SUTVA violation), and social
features spill across the graph (treated users message control users). Cures: randomize
clusters (geos, seller pools, subgraphs) — pay variance for independence.

**5.** Sliding window: adapts fast, amputates seasons. Full history: remembers December,
dilutes the recent world (and drags pre-concept-drift garbage back in — our §1 retrains
deliberately used windows). Better: month/season as features with full (post-drift)
history, or per-season models the router chooses. The enabler: the date-partitioned
immutable lake (nb 14) — every windowing policy is just a different glob over
partitions, so *try all three* (nb 17's tracker decides).

## 4. Takeaways

- Retraining is a *policy*: never (fossilizes), scheduled (simple, sometimes wasteful),
  triggered (efficient, needs nb 19's monitors). All are floored by **label delay +
  data accumulation** — invest in shortening the loop before sharpening the model.
- Test in production up a ladder of risk: **shadow** (free, catches everything except
  behavior), **canary** (bounded blast radius + strict tripwires + instant rollback),
  **A/B** (the measuring instrument — pre-commit the horizon or use sequential methods;
  peeking manufactures ~25% false winners), **bandits** (cheap decisions, biased
  measurements — perfect for many fast variants and exploration budgets).
- Keep A/A tests running; randomize the unit that experiences the treatment; guard the
  metrics that must never lose.
- The loop is now closed: monitor (19) → retrain (20 §1) → validate progressively
  (20 §2) → serve (18) → monitor... Notebook 21 builds the machine that turns this loop
  from heroics into a cron entry.

**Further reading.** DMLS ch. 9; Kohavi, Tang & Xu, *Trustworthy Online Controlled
Experiments* (2020) — the A/B bible, peeking included; Russo et al., *A Tutorial on
Thompson Sampling* (2018).

**Next:** [21 · Infrastructure & MLOps](21-infra-mlops.ipynb) — Part 2's finale: the
orchestrator, registry, and platform that run everything you've built while you sleep.